# Comparative Results Summary

This notebook loads `experiments/full_evaluation.json` and creates a compact comparison table for all saved models.

It focuses on the most relevant metrics for the fraud detection project:
- PR-AUC
- F1
- Recall
- NLL
- Brier Score
- ECE


In [ ]:
from pathlib import Path
import json
import pandas as pd

RESULTS_PATH = Path('experiments/full_evaluation.json')

if not RESULTS_PATH.exists():
    raise FileNotFoundError(
        f'Could not find results file at: {RESULTS_PATH}\n'
        'Run the evaluation script first, or update RESULTS_PATH.'
    )

with RESULTS_PATH.open('r', encoding='utf-8') as f:
    results = json.load(f)

print('Loaded:', RESULTS_PATH)


In [ ]:
rows = []

for model_name, model_info in results['models'].items():
    for split_name in ['validation', 'test']:
        split_info = model_info[split_name]
        cls = split_info['classification_metrics']
        prob = split_info['proper_scoring_rules']
        cal = split_info['calibration_metrics']

        rows.append({
            'model': model_name,
            'split': split_name,
            'pr_auc': cls['pr_auc'],
            'f1': cls['f1'],
            'recall': cls['recall'],
            'precision': cls['precision'],
            'roc_auc': cls['roc_auc'],
            'nll': prob['nll'],
            'brier_score': prob['brier_score'],
            'ece': cal['ece'],
            'mce': cal['mce'],
        })

df = pd.DataFrame(rows)
df.head()


In [ ]:
# Compact table focused on the test split
test_df = df[df['split'] == 'test'].copy()
test_df = test_df.sort_values(by=['pr_auc', 'f1', 'nll'], ascending=[False, False, True])

compact_cols = [
    'model', 'pr_auc', 'f1', 'recall', 'precision', 'nll', 'brier_score', 'ece'
]

compact_table = test_df[compact_cols].reset_index(drop=True)
compact_table.style.format({
    'pr_auc': '{:.4f}',
    'f1': '{:.4f}',
    'recall': '{:.4f}',
    'precision': '{:.4f}',
    'nll': '{:.4f}',
    'brier_score': '{:.4f}',
    'ece': '{:.4f}',
})


In [ ]:
# Validation vs test side by side
pivot_table = df.pivot(index='model', columns='split', values=['pr_auc', 'f1', 'recall', 'nll', 'brier_score', 'ece'])
pivot_table = pivot_table.sort_values(by=('pr_auc', 'test'), ascending=False)
pivot_table.style.format('{:.4f}')


In [ ]:
# Optional: export compact table to CSV
output_csv = Path('experiments/model_comparison_table.csv')
compact_table.to_csv(output_csv, index=False)
print(f'Saved compact comparison table to: {output_csv}')
